In [33]:
import genbrain_model_3dot as model
from genbrain_3dot_smc import collect_frames
from genbrain_smcnn_core.interpreter import run_smcnn_particle_filter
from genbrain_3dot_smc.viz import *
import genbrain_utils_genjax as gjutils
import jax.numpy as jnp
import jax
import genstudio.plot as Plot


In [2]:
# First we will collect the generative functions from the 3dot model. 
genfns = [
    model.initial_proposal,
    model.initial_model,
    model.step_proposal,
    model.step_model,
    model.obs_model,
]

In [3]:
# Here we convert digital x,y numpy frames into a spherical occupancy map (i.e. a visual angle occupancy grid)
xy_obs_frames = collect_frames()
vis_angle_observations = jax.vmap(lambda obs: model.find_occupied_2d_angles(obs))(
    jnp.array(xy_obs_frames)
)
len_sim = 5
obs_traces = model.generate_obs_traces(vis_angle_observations[0:len_sim])

In [4]:
# choose a random seed so you can repeat the experiment
key = jax.random.PRNGKey(100)
num_particles = 20

init_states_and_scores, first_step_states_and_scores, unrolled_pf = (
    gjutils.smc.run_particle_filter(
        obs_traces,
        num_particles,
        len_sim,
        genfns,
        key,
        model.translate_proposal_cm_to_model,
    )
)
# keep print of scores or not? its useful for debugging.

Values: P = [-inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf
 -inf -inf -inf -inf -inf -inf], Q = [ -8.614116 -12.240436  -8.066771  -7.606756 -10.532404 -10.07239
 -14.469076 -10.72326  -10.532404  -8.154101 -14.343352  -8.614116
  -8.154101 -10.263247 -10.532404 -10.72326   -7.606756 -10.72326
 -10.072391  -7.606756], O = [-31.190197 -34.666298 -34.666298 -34.666298 -38.142395 -38.142395
 -41.618492 -34.666298 -38.142395 -31.190197 -38.142395 -31.190197
 -31.190197 -34.666298 -38.142395 -34.666298 -34.666298 -34.666298
 -38.142395 -34.666298]
Values: P = [      -inf       -inf       -inf       -inf       -inf       -inf
       -inf -15.820416       -inf       -inf -31.632635       -inf
       -inf       -inf       -inf       -inf       -inf       -inf
       -inf       -inf], Q = [-3.0486007 -5.619448  -5.639096  -5.3669643 -4.9069514 -9.021802
 -5.6194477 -6.0991087 -3.754869  -3.0486007 -5.3928404 -5.140358
 -3.5086145 -3.5086145 -3.754869  -7.0177836 -5.86570

In [5]:
latent_variables = [
    {
        "variable": "v3d",
        "q_id": ("dot", "v3d"),
        "q_parents": [],
        "p_parents": [],
        "support": model.xyz_vels,
        "type": "distribution",
    },
    {
        "variable": "xyz",
        "q_id": ("dot", "xyz"),
        "q_parents": [("ego_pos", "ego_matter")],
        "p_parents": [],
        "support": model.xyz_point_cloud,
        "type": "distribution",
    },
    {
        "variable": ("ego_pos", "ego_matter"),
        "q_id": ("dot", "ego_pos", "ego_matter"),
        "q_parents": [],
        "p_parents": ["xyz"],
        "support": model.bool_support,
        "type": "probmap",
    },
    {
        "variable": "lights",
        "q_id": ("dot", "lights"),
        "p_parents": [],
        "q_parents": [],
        "support": model.bool_support,
        "type": "distribution",
    },
    {
        "variable": "diam",
        "q_id": ("dot", "diam"),
        "p_parents": [],
        "q_parents": [],
        "support": model.diams,
        "type": "distribution",
    },
]

obs_variables = [
    {
        "variable": ("obs", "pix"),
        "parents": [],
        "support": model.bool_support,
        "type": "probmap",
    }
]




In [6]:
results = run_smcnn_particle_filter(
    (latent_variables, obs_variables),
    model.initial_model,
    model.step_model,
    model.initial_proposal,
    model.step_proposal,
    model.obs_model,
    5,
    2,
    vis_angle_observations[1:3],
)


Jitting Generative Functions
Initializing SMCNN Particle Filter
Values: P = [      -inf       -inf       -inf -14.069142       -inf       -inf
 -31.47956  -32.92604        -inf -18.1999   -15.98837  -13.67204
 -14.069142 -20.997154 -12.622659 -24.054615       -inf       -inf
       -inf       -inf], Q = [ -3.754869   -3.754869   -9.319347   -5.619448   -7.2321143 -10.469468
  -9.362024   -7.25119    -3.6446338  -5.6194477  -5.159434   -5.6194477
  -5.619448   -7.711205   -5.6194477  -7.0030193  -4.8921866  -3.508615
  -3.508615   -5.140358 ], O = [-34.666298 -34.666298 -38.142395 -38.142395 -38.142395 -31.190197
 -48.57069  -41.618492 -38.142395 -38.142395 -38.142395 -38.142395
 -38.142395 -41.618492 -38.142395 -38.142395 -31.190197 -31.190197
 -31.190197 -34.666298]
Values: P = [-15.793759 -18.565975       -inf       -inf       -inf       -inf
       -inf       -inf       -inf -12.996509       -inf -22.79751
       -inf       -inf -12.996509       -inf -12.025179 -21.972832
       -in

In [7]:
xyz_spikes = gather_spikes_from_single_particle_samplescore(results, 0, "xyz", range(len_sim))
x_p_assemblies_particle_0 = snmc_spikes_wrapper(
    results, 'xyz', range(5), range(0,1), ["assemblies_p13", "assemblies_p14", "assemblies_p15"], True)[0]

<IPython.core.display.Javascript object>

In [21]:
xyz_filtered = {k: v for k, v in xyz_spikes[0].items() if len(v[0]) != 0}

In [22]:
xyz_filtered

{168: (array([19.48261399]), 'assemblies_p1176'),
 245: (array([31.59790063]), 'assemblies_p1160'),
 505: (array([38.08324293]), 'assemblies_p1108'),
 622: (array([15.28173786]), 'assemblies_p1085'),
 722: (array([31.94937125]), 'assemblies_p1065'),
 945: (array([17.51314819]), 'assemblies_p1020'),
 1228: (array([43.74575028]), 'assemblies_p964'),
 1287: (array([36.98844316]), 'assemblies_p952'),
 1291: (array([43.00687758]), 'assemblies_p951'),
 1306: (array([39.23915585]), 'assemblies_p948'),
 1560: (array([24.01980291]), 'assemblies_p897'),
 1563: (array([33.24402964]), 'assemblies_p897'),
 1564: (array([32.23495927]), 'assemblies_p897'),
 1610: (array([31.16554572]), 'assemblies_p887'),
 1700: (array([45.95304026]), 'assemblies_p869'),
 1717: (array([28.83537722]), 'assemblies_p866'),
 2041: (array([20.89011814]), 'assemblies_p801'),
 2235: (array([10.12631173]), 'assemblies_p762'),
 2385: (array([12.70209617]), 'assemblies_p732'),
 2414: (array([21.11396159]), 'assemblies_p727'),


In [23]:
static_plot_snmc([xyz_filtered])

<IPython.core.display.Javascript object>

In [ ]:
def plot_spike_dictionary(spiketimes_dict):
    fig, ax = plt.subplots()
    cpal = my_tab20(100)
    num_components = len(spiketimes_dict)
    for neuron_id, spikes_and_label in enumerate(spiketimes_dict.values()):
        spikes, label = spikes_and_label
        neuron_y = num_components - (neuron_id + 1)
        print(neuron_y) 
        print(spikes)
        ax.vlines(
                spikes, neuron_y, neuron_y + 0.8, color='k', linewidth=1.0
            )
    comp_labels = [v[1] for k, v in spiketimes_dict.items()]
    comp_labels.reverse()
    ax.set_ylim(0.5, len(comp_labels) + 0.5)
    ax.set_yticks(range(len(comp_labels)))
    ax.set_yticklabels(comp_labels, fontdict={"fontsize": 5})
    ax.set_xlabel("Time")
    ax.set_ylabel("Neuron ID")
    ax.set_title("Raster Plot")
    plt.grid(False)
    plt.tight_layout()
    plt.show()


In [44]:
def plot_spike_dictionary_studio(spiketimes_dict):
    num_components = len(spiketimes_dict)
    plot_objects = []
    for neuron_id, spikes_and_label in enumerate(spiketimes_dict.values()):
        spikes, label = spikes_and_label
        neuron_y = num_components - (neuron_id + 1)
        for sp in spikes:
            plot_objects.append({"X": sp, "Y": neuron_y, "CATEGORY": label})
    comp_labels = [v[1] for k, v in spiketimes_dict.items()]
    comp_labels.reverse()
    #Plot.dot(plot_objects, x="X", y="Y", fill="CATEGORY", r=20)
    Plot.tickX(plot_objects, x="X", y="y")
    return plot_objects

In [45]:
pa = plot_spike_dictionary_studio(xyz_filtered)

In [46]:
pa

[{'X': 19.48261399345018, 'Y': 134, 'CATEGORY': 'assemblies_p1176'},
 {'X': 31.597900632257534, 'Y': 133, 'CATEGORY': 'assemblies_p1160'},
 {'X': 38.083242930757386, 'Y': 132, 'CATEGORY': 'assemblies_p1108'},
 {'X': 15.281737861384205, 'Y': 131, 'CATEGORY': 'assemblies_p1085'},
 {'X': 31.949371249194535, 'Y': 130, 'CATEGORY': 'assemblies_p1065'},
 {'X': 17.5131481933529, 'Y': 129, 'CATEGORY': 'assemblies_p1020'},
 {'X': 43.7457502847426, 'Y': 128, 'CATEGORY': 'assemblies_p964'},
 {'X': 36.9884431583019, 'Y': 127, 'CATEGORY': 'assemblies_p952'},
 {'X': 43.00687757592974, 'Y': 126, 'CATEGORY': 'assemblies_p951'},
 {'X': 39.239155852207375, 'Y': 125, 'CATEGORY': 'assemblies_p948'},
 {'X': 24.019802908464538, 'Y': 124, 'CATEGORY': 'assemblies_p897'},
 {'X': 33.244029641341754, 'Y': 123, 'CATEGORY': 'assemblies_p897'},
 {'X': 32.234959265502496, 'Y': 122, 'CATEGORY': 'assemblies_p897'},
 {'X': 31.165545721849142, 'Y': 121, 'CATEGORY': 'assemblies_p887'},
 {'X': 45.953040259100916, 'Y': 120,